In [40]:
import os
import json
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
from SoccerNet.utils import getListGames
from torchvision.ops import focal_loss


In [62]:
class SoccerNetActionDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.root_dir = root_dir
        self.split = split
        self.games = getListGames(split=split)
        self.data_map = []

        # Label to ID mapping (Class 0 is reserved for 'Background')
        self.label_to_id = {
            "Background": 0,
            "Ball out of play": 1,
            "Clearance": 2,
            "Corner": 3,
            "Direct free-kick": 4,
            "Foul": 5,
            "Goal": 6,
            "Indirect free-kick": 7,
            "Injury": 8,
            "Kick-off": 9,
            "Offside": 10,
            "Penalty": 11,
            "Red card": 12,
            "Shots off target": 13,
            "Shots on target": 14,
            "Substitution": 15,
            "Throw-in": 16,
            "Yellow card": 17
        }

        self._build_index()

    def _build_index(self):
        print(f"Indexing {self.split} set...")
        count = 0
        for game in self.games:
            game_path = os.path.join(self.root_dir, game)
            json_path = os.path.join(game_path, "Labels-v2.json")
            video_path = os.path.join(game_path, "1_224p.mkv")

            if os.path.exists(json_path) and os.path.exists(video_path):
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    for annotation in data['annotations']:
                        if annotation['label'] in self.label_to_id:
                            self.data_map.append({
                                'label': annotation['label'],
                                'gameTime': annotation['gameTime'],
                                'video_path': video_path,
                            })
                            count += 1
        #print(f"Safely indexed {count} action points backed by downloaded videos in {self.split} split.")

    def _load_video_clip(self, video_path, frame_idx, num_frames=32):
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

        frames = []
        for _ in range(num_frames):
            ret, frame = cap.read()
            if not ret:
                # Pad with the last successful frame or zeros
                frame = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                frame = cv2.resize(frame, (224, 224))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        # Convert to Tensor [Channels, Time, Height, Width]
        clip = np.array(frames).astype(np.float32) / 255.0
        clip = clip.transpose(3, 0, 1, 2)
        return torch.from_numpy(clip)

    def __len__(self):
        return len(self.data_map)

    def __getitem__(self, idx):
        sample = self.data_map[idx]

        # 1. Parse Time
        try:
            parts = sample['gameTime'].split(' - ')
            timestamp = parts[1] if len(parts) > 1 else parts[0]
            minutes, seconds = map(float, timestamp.split(':'))
            total_seconds = minutes * 60 + seconds
        except:
            total_seconds = 0.0

        # 2. Frame Calculation
        fps = 25.0
        start_frame = max(0, int((total_seconds - 1.0) * fps))

        # 3. Stream Video
        video_tensor = self._load_video_clip(sample['video_path'], start_frame, num_frames=32)

        # 4. SlowFast Split
        # Fast path: All 32 frames. Slow path: Every 4th frame (8 total).
        fast_path = video_tensor
        slow_path = video_tensor[:, ::4, :, :]

        # 5. Label
        label_id = self.label_to_id.get(sample['label'], 0)

        return [slow_path, fast_path], torch.tensor(label_id, dtype=torch.long)

In [68]:
getListGames(split="train")[0]


'england_epl\\2014-2015\\2015-02-21 - 18-00 Chelsea 1 - 1 Burnley'

In [69]:
# 2. THE MODEL (Standardized Hub Loading)
def get_model(num_classes):
    # Using Torch Hub for reproducible architecture and pre-trained weights
    model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)
    # Surgical replacement of the final head
    model.blocks[-1].proj = nn.Linear(model.blocks[-1].proj.in_features, num_classes)
    return model

In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # 1. Calculate standard cross entropy log probabilities
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # 2. Get the probability of the correct class
        pt = torch.exp(-ce_loss)

        # 3. Apply the focal loss mathematical formula: (1 - pt)^gamma
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import wandb

# Define your root path as a raw string to handle backslashes correctly
ROOT_PATH = r"C:\Users\omoni\Desktop\experiments\Action_recognition_slowfast\experiment_2\SoccerNet_Dataset"

# 1. Correct Sweep Configuration
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_loss', 'goal': 'minimize'},
    'parameters': {
        'batch_size': {'values': [4, 8]},
        'learning_rate': {'values': [0.001, 0.0001]},
        'epochs': {'values': [2, 4]},
        "architecture": {'value': "SlowFast"},
        'loss_type': {'values': ["cross_entropy", "focal_loss"]},
        'action_weight_multiplier': {'values': [4.0, 10.0, 15.0]},
        'focal_gamma': {'values': [1.5, 2.0, 3.0]},
        "dataset": {'value': "SoccerNet-v2"}
    }
}

sweep_id = wandb.sweep(sweep_config, project="slow_fast_action_spotting_model_experiment_2")

# 2. Validation Helper (Fixed: Reformats inputs to the strict dictionary schema PyTorchVideo requires)
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Unpack the list sent from your dataset __getitem__
            slow_tensor, fast_tensor = inputs

            # Form the structure PyTorchVideo expects: a list containing a dict
            formatted_input = [{"slow": slow_tensor.to(device), "fast": fast_tensor.to(device)}]

            outputs = model(formatted_input)
            loss = criterion(outputs, targets.to(device))
            val_loss += loss.item()
    return val_loss / len(val_loader)

# 3. Training Logic (Fixed: Reformats inputs to avoid the silent freezing bottleneck)
def train_model(model, train_loader, val_loader, optimizer, scheduler, num_epochs, criterion, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            optimizer.zero_grad()

            # Unpack the list sent from your dataset __getitem__
            slow_tensor, fast_tensor = inputs

            # Form the structure PyTorchVideo expects: a list containing a dict
            formatted_input = [{"slow": slow_tensor.to(device), "fast": fast_tensor.to(device)}]

            outputs = model(formatted_input)
            loss = criterion(outputs, targets.to(device))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            wandb.log({"batch_loss": loss.item()})

            # Print feedback loop so you see active progress step-by-step
            if batch_idx % 5 == 0:
                print(f"Epoch {epoch} | Batch {batch_idx}/{len(train_loader)} | Current Loss: {loss.item():.4f}")

        # Validation Phase
        val_loss = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        # Log Epoch Metrics
        wandb.log({"epoch": epoch, "train_loss": total_loss / len(train_loader), "val_loss": val_loss})
        print(f"--- Epoch {epoch} Complete | Avg Train Loss: {total_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f} ---")

        # Save Checkpoint
        torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")

# 4. Main Train Entry Point Wrapper
def train():
    with wandb.init() as run:
        config = wandb.config
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"\n--- Initializing Active Sweep Agent Run on Device: {device} ---")

        # Load 18-class model head structure
        model = get_model(num_classes=18).to(device)

        # CRITICAL SAFETY FIX: Freeze the massive backbone body to avoid OOM memory crashes
        for param in model.parameters():
            param.requires_grad = False
        # Un-freeze only the new linear classifier projection layers we just replaced
        for param in model.blocks[-1].proj.parameters():
            param.requires_grad = True

        # Setup DataLoaders (using your active SoccerNetActionDataset class in notebook memory)
        train_ds = SoccerNetActionDataset(root_dir=ROOT_PATH, split="train")
        val_ds = SoccerNetActionDataset(root_dir=ROOT_PATH, split="valid")

        train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=0)

        # Create 18-element class penalty weights array
        weight = torch.ones(18).to(device)
        weight[1:] = config.action_weight_multiplier

        # Target optimization parameters strictly to the classification head
        trainable_parameters = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.Adam(trainable_parameters, lr=config.learning_rate)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1)

        # Dynamically assign loss configurations safely
        if config.loss_type == "cross_entropy":
            criterion = nn.CrossEntropyLoss(weight=weight)
        elif config.loss_type == "focal_loss":
            criterion = FocalLoss(gamma=config.focal_gamma)
        else:
            criterion = nn.CrossEntropyLoss()

        # Run training cycle execution
        train_model(model, train_loader, val_loader, optimizer, scheduler, config.epochs, criterion, device)

# 5. Define and Launch the Active Agent Space Tracker
wandb.agent(sweep_id, function=train)


Create sweep with ID: s6fxvvnw
Sweep URL: https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spotting_model_experiment_2/sweeps/s6fxvvnw


wandb: Agent Starting Run: 58zyb1bz with config:
wandb: 	action_weight_multiplier: 4
wandb: 	architecture: SlowFast
wandb: 	batch_size: 4
wandb: 	dataset: SoccerNet-v2
wandb: 	epochs: 2
wandb: 	focal_gamma: 1.5
wandb: 	learning_rate: 0.001
wandb: 	loss_type: cross_entropy
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\omoni\_netrc.
wandb: setting up run 58zyb1bz
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in C:\Users\omoni\Desktop\experiments\crystal_palace_action_detection\wandb\run-20260622_233957-58zyb1bz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run apricot-sweep-1
wandb:  View project at https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spotting_model_experiment_2
wandb:  View sweep at https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spotting_model_experiment_2/sweeps/s6fxvvnw
wandb:  View run at https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spot

Indexing train set...
Safely indexed 3319 action points backed by downloaded videos in train split.
Indexing valid set...
Safely indexed 985 action points backed by downloaded videos in valid split.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: batch_loss ▄▅▅▄▃▄▄▃▃█▅▁▁▆▂▃▄▄▄▆▆▃▇▃▁▂▄▃▃▅▃▅▃▃▁▂▃▃▃▄
wandb:      epoch ▁█
wandb: train_loss █▁
wandb:   val_loss ▁█
wandb: 
wandb: Run summary:
wandb: batch_loss 2.42312
wandb:      epoch 1
wandb: train_loss 2.23763
wandb:   val_loss 2.29869
wandb: 
wandb:  View run apricot-sweep-1 at: https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spotting_model_experiment_2/runs/58zyb1bz
wandb:  View project at: https://wandb.ai/25817442-edge-hill-university/slow_fast_action_spotting_model_experiment_2
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: .\wandb\run-20260622_233957-58zyb1bz\logs
wandb: Agent Starting Run: u22h5b62 with config:
wandb: 	action_weight_multiplier: 4
wandb: 	architecture: SlowFast
wandb: 	batch_size: 4
wandb: 	dataset: SoccerNet-v2
wandb: 	epochs: 2
wandb: 	focal_gamma: 1.5
wandb: 	l

Indexing train set...
Safely indexed 3319 action points backed by downloaded videos in train split.
Indexing valid set...
Safely indexed 985 action points backed by downloaded videos in valid split.
